In [414]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

In [415]:
data = pd.read_csv('AmesHousing.csv')

In [416]:
drop_cols = [
    'Garage Type',
    'Mas Vnr Type',
    'Alley',
    'Fence',
    'Misc Feature',
    'Order',
    'PID',
    'Utilities'
]

In [417]:
data = data.drop(columns=drop_cols)

In [418]:
X = data.drop('SalePrice', axis=1)
y = data['SalePrice']

In [419]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Columns divided as per required missing strategy

In [420]:
missing_none = ['Pool QC', 'Fireplace Qu', 'Bsmt Qual', 'Bsmt Cond', 'Garage Finish', 'Garage Qual', 
                'Garage Cond', 'BsmtFin Type 2', 'BsmtFin Type 1']

In [421]:
most_frequent = ['Electrical', 'Garage Cars']

In [422]:
as_zero = ['Bsmt Full Bath', 'Bsmt Half Bath', 'BsmtFin SF 1', 'BsmtFin SF 2', 'Bsmt Unf SF', 'Total Bsmt SF', 'Mas Vnr Area']

In [423]:
median_col = ['Lot Frontage', 'Garage Yr Blt', 'Garage Area']

In [424]:
bsmt_col = ['Bsmt Exposure']

### Handling Missing Values

In [425]:
def imputing_missing_values(X_train, X_test):

    #Filling with None
    X_train[missing_none] = X_train[missing_none].fillna('None')
    X_test[missing_none] = X_test[missing_none].fillna('None')

    #Filling with most frequent
    for col in most_frequent:
        mode = X_train[col].mode()[0]
        X_train[col] = X_train[col].fillna(mode)
        X_test[col] = X_test[col].fillna(mode)

    #Filling with zero
    X_train[as_zero] = X_train[as_zero].fillna(0)
    X_test[as_zero] = X_test[as_zero].fillna(0)

    #Filling with median
    for col in median_col:
        median_value = X_train[col].median()
        X_train[col] = X_train[col].fillna(median_value)
        X_test[col] = X_test[col].fillna(median_value)

    #Filling Bsmt column
    X_train[bsmt_col] = X_train[bsmt_col].fillna('No Basement')
    X_test[bsmt_col] = X_test[bsmt_col].fillna('No Basement')

    return X_train, X_test

In [426]:
X_train, X_test = imputing_missing_values(X_train, X_test)

In [427]:
X_train.isnull().sum().sum()

np.int64(0)

### Ordinal columns 

In [428]:
qual_with_none_columns = [
    'Bsmt Qual',
    'Bsmt Cond',
    'Pool QC',
    'Fireplace Qu',
    'Garage Qual',
    'Garage Cond'
]

In [429]:
quality_columns = [
    'Exter Qual',
    'Exter Cond',
    'Kitchen Qual',
    'Heating QC',
]

In [430]:
functional_column = ['Functional']
garage_column = ['Garage Finish']
paved_column = ['Paved Drive']
bsmt_exposure_column = ['Bsmt Exposure']
finish_column = ['BsmtFin Type 1', 'BsmtFin Type 2']

### Ordinal Encoding Columns

In [431]:
ordinal_categories = {
    'quality': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'none_quality': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'garage_finish': ['None', 'Unf', 'RFn', 'Fin'],
    'functional': ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ'],
    'paved_drive': ['N', 'P', 'Y'],
    'bsmt_exposure': ['No Basement', 'No', 'Mn', 'Av', 'Gd'],
    'finish': ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ']
}

### One Hot Encoding Columns

In [432]:
nominal_cols = ['Neighborhood', 'MS Zoning', 'House Style', 'Condition 1', 'Condition 2', 'Bldg Type', 'Street', 
                'Lot Config', 'Roof Style', 'Exterior 1st', 'Exterior 2nd', 'Foundation', 'Heating', 'Sale Type', 
                'Sale Condition', 'Lot Shape', 'Land Contour', 'Land Slope', 'Roof Matl', 'Central Air', 'Electrical']

### Encoding

In [433]:
encoder = ColumnTransformer(
    transformers = [
        (
            'quality',
            OrdinalEncoder(
                categories = [ordinal_categories['quality']] * len(quality_columns)
            ),
            quality_columns
        ),
        
        (
            'none_column_quality',
            OrdinalEncoder(
                categories = [ordinal_categories['none_quality']] * len(qual_with_none_columns)
            ),
            qual_with_none_columns
        ),
        
        (
            'garage_finish',
            OrdinalEncoder(
                categories = [ordinal_categories['garage_finish']]
            ),
            garage_column
        ),
        
        (
            'functional',
            OrdinalEncoder(
                categories = [ordinal_categories['functional']]
            ),
            functional_column
        ),
        
        (
            'paved_drive',
            OrdinalEncoder(
                categories = [ordinal_categories['paved_drive']] 
            ),
            paved_column
        ),
        
        (
            'basement_exposure',
            OrdinalEncoder(
                categories = [ordinal_categories['bsmt_exposure']]
            ),
            bsmt_exposure_column
        ),

        (
            'finish',
            OrdinalEncoder(
                categories = [ordinal_categories['finish']] * len(finish_column)
            ),
            finish_column
        ),
        
        (
            'nominal_encoding',
            OneHotEncoder(handle_unknown='ignore', sparse_output=False),
            nominal_cols
        )
], remainder='passthrough')

### Fitting the encoder to data

In [434]:
X_train_transform = encoder.fit_transform(X_train)
X_test_transform = encoder.transform(X_test)

In [435]:
lr = LinearRegression()

In [436]:
lr.fit(X_train_transform, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [437]:
y_pred = lr.predict(X_test_transform)

In [438]:
RMSE = root_mean_squared_error(y_test, y_pred)
MAE = mean_absolute_error(y_test, y_pred)
R2 = r2_score(y_test, y_pred)

In [439]:
print('RMSE: ', RMSE)
print('MAE: ', MAE)
print('R2: ',R2)

RMSE:  30656.434309583463
MAE:  18079.079909947508
R2:  0.8827800004843378
